## Importing Libraries

In [30]:
import os
import math
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import create_agent                      # ✅ new API
from langgraph.checkpoint.memory import MemorySaver            # ✅ replaces manual chat_history



load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

### DEFINE TOOLS

In [14]:
@tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    Input must be a valid Python math expression as a string.
    Examples: '2 + 2', '15 * 4', 'math.sqrt(144)', '100 / 7'
    Use this for any arithmetic or math calculations."""
    try:
        # Safely evaluate the expression using eval with limited built-ins
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error evaluating '{expression}': {str(e)}"

@tool
def get_string_info(text: str) -> str:
    """Analyzes a string and returns character count, word count, and whether it's a palindrome.
    Use this when asked about properties of text or strings."""
    char_count = len(text)
    word_count = len(text.split())
    char_no_spaces = len(text.replace(" ", ""))
    is_palindrome = text == text[::-1]
    return (
        f"Text: '{text}'\n"
        f"Character count (with spaces): {char_count}\n"
        f"Character count (no spaces): {char_no_spaces}\n"
        f"Word count: {word_count}\n"
        f"Is palindrome: {is_palindrome}"
    )

@tool
def get_backend_fact(topic: str) -> str:
    """Returns a key fact about a backend engineering topic.
    Supported topics: redis, postgresql, docker, kubernetes, jwt, fastapi.
    Use this when asked about backend engineering concepts."""
    facts = {
        "redis": "Redis is an in-memory data structure store. Default max memory is determined by available RAM. Use TTL on all cached keys to prevent memory exhaustion.",
        "postgresql": "PostgreSQL uses MVCC (Multi-Version Concurrency Control) for transaction isolation. Each transaction sees a snapshot of data from when it started.",
        "docker": "Docker containers share the host OS kernel but have isolated filesystems, networks, and process spaces via Linux namespaces and cgroups.",
        "kubernetes": "Kubernetes schedules containers across nodes using the kube-scheduler. Pods are the smallest deployable unit — one or more containers sharing network and storage.",
        "jwt": "JWT access tokens should be short-lived (15-60 min). Refresh tokens should be rotated on each use. Store refresh tokens in the database to enable revocation.",
        "fastapi": "FastAPI uses Python type hints + Pydantic for automatic request validation and OpenAPI doc generation. Async endpoints use asyncio for non-blocking I/O.",
    }
    topic_lower = topic.lower()
    return facts.get(topic_lower, f"No fact available for topic '{topic}'. Supported topics: {', '.join(facts.keys())}.")

tools = [calculate, get_string_info, get_backend_fact]

### BUILD AGENT PROMPT

*agent_scratchpad is REQUIRED — it's where the ReAct loop (Thought/Action/Observation)*

*gets recorded so the LLM can see its own previous steps*

In [23]:
prompt = """You are a helpful backend engineering assistant.
You have access to tools for calculations, string analysis, and backend facts.
Always use tools when they're relevant to the question.
Think step by step before answering complex questions."""


### CREATE AGENT + EXECUTOR
*create_tool_calling_agent: returns a Runnable (not yet executable)*

*It uses the LLM's native function-calling capability to select tools*

*AgentExecutor: runs the ReAct loop*

*max_iterations: safety limit to prevent infinite loops*

*handle_parsing_errors: gracefully handles malformed LLM output*

In [31]:
checkpointer = MemorySaver()

agent = create_agent(model=llm, tools=tools, system_prompt=prompt, checkpointer=checkpointer)  # ✅ new API with built-in memory management

### SINGLE-TURN USAGE

In [33]:
print("=== Single-Turn Agent ===\n")

result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "How many characters are in 'PostgreSQL', and what is that number squared?"}]},
    config={"configurable": {"thread_id": "single-1"}}
)

print(f"Final Answer: {result1['messages'][-1].content}\n")

print("="*60)

result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What should I know about JWT tokens for backend security?"}]},
    config={"configurable": {"thread_id": "single-2"}}
)

print(f"Final Answer: {result2['messages'][-1].content}\n")

print("="*60)

=== Single-Turn Agent ===

Final Answer: - Number of characters in 'PostgreSQL': 10
- 10 squared: 100

Final Answer: Here’s a practical overview of JWTs for backend security, focusing on solid, common patterns you can implement today.

Key ideas
- What JWTs are: a compact, signed (and sometimes encrypted) token that carries a set of claims (like user ID, roles, expiry) in a payload, with a signature to verify integrity.
- Typical usage: the client sends the JWT in an Authorization header as a Bearer token (or in a secure, HttpOnly cookie). The server validates the signature and the claims before granting access.
- The goal: prove who you are and what you’re allowed to do, without storing session state on the server.

Recommended lifecycle and token types
- Access token: short-lived (usually 15–60 minutes). This is the token used to access APIs.
- Refresh token: longer-lived, but not sent to every API call. It’s used to obtain new access tokens when the old one expires.
- Rotate refresh

### MULTI-TURN CHAT WITH MEMORY

In [34]:
print("\n=== Multi-Turn Agent with Memory ===\n")

THREAD_ID = "multi-turn-1"

def chat_with_agent(user_input: str) -> str:
    """Run one turn of conversation, maintaining history across calls."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": THREAD_ID}}
        )
    
    return result["messages"][-1].content

r1 = chat_with_agent("My SaaS API uses JWT tokens with a 60 minute expiry.")
print(f"Agent turn 1: {r1}\n")

r2 = chat_with_agent("Is that expiry time a good choice?")
print(f"Agent turn 2(references previous): {r2}\n")

r3 = chat_with_agent("What is 60 multiplied by 24 multiplied by 7?")
print(f"Agent turn 3(tool usage): {r3}\n")

state = agent.get_state(config={"configurable": {"thread_id": THREAD_ID}})
print(f"Total messages in thread memory: {len(state.values['messages'])}")


=== Multi-Turn Agent with Memory ===

Agent turn 1: Nice. You’re within the commonly recommended window for access tokens (15–60 minutes). Given security concerns with JWTs, here’s a solid approach to improve safety and revocation without sacrificing usability:

Key recommendations
- Use short-lived access tokens (30–60 minutes is fine; 15–30 minutes is safer if you can handle refresh).
- Implement refresh token rotation: issue a new refresh token every time the client uses one, and invalidate the old refresh token.
- Store refresh tokens server-side (ideally hashed) to enable revocation and detect misuse.
- Include a unique identifier (jti) in tokens to help detect reused tokens.
- Use a strong signing method (RS256 with a rotating public key, or HS256 with a secure key management plan). Consider public/private key (RS256) for easier key rotation.
- Validate all tokens thoroughly: issuer (iss), audience (aud), subject (sub), expiry (exp), not-before (nbf), and the jti for revocation 